# 직접 해보기 03 · 33분에서 β까지

논문이 공개한 표의 값(Cheng 외 2023, Thomas 외 2023)으로 DART의 결과를 다시 계산한다. 관측 원자료가 아니라 표의 숫자다.

같은 계산이 `analysis.py`에도 들어 있다.

In [ ]:
import analysis as A
v = A.load()
print('파일 지문', A.sha8())
r = A.compute(v)
print(f"ΔP {r['dP_min']:.1f}분 · v {r['v_cm_s']:.1f}cm/s · Δv {r['dv_mm_s']:.2f}mm/s")

## 1. 옛 주기로 예측한 그림자와 실제 그림자의 어긋남
matplotlib이 없으면 `pip install matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
P1 = v['P_post']; dP = (v['P_pre'] - P1) * 60
fig, ax = plt.subplots(figsize=(8, 3.5))
d = [14 * k / 100 for k in range(101)]
ax.plot(d, [x * 24 / P1 * dP for x in d])
ax.axvspan(1.23, 14, color='gray', alpha=.15)
ax.set_xlabel('충돌 뒤 날수'); ax.set_ylabel('앞당겨진 시각(분)'); plt.tight_layout()

## 2. β를 밀도의 함수로 그린다 (책의 예상 결과: 2.19 · 3.50 · 4.81)

In [ ]:
rhos = list(range(1000, 4001, 50))
plt.figure(figsize=(8, 3.5))
plt.plot(rhos, [r['beta_fn'](x) for x in rhos])
plt.axhline(1, ls='--', color='k'); plt.scatter([2400], [3.61], marker='s', color='k')
plt.xlabel('가정한 밀도 (kg/m³)'); plt.ylabel('β')
for x in (1500, 2400, 3300): print(x, round(r['beta_fn'](x), 2))

## 3. 조건을 바꿔 본다
충돌 뒤 주기나 궤도 반지름을 바꾸면 Δv가 어떻게 달라지는가? 내가 구한 Δv로 β를 계산하면(`use_paper_dv=False`) 3.50은 어떻게 바뀌는가? 바꾼 조건과 결과를 연구 노트에 적어 두자.

In [ ]:
for p in (11.35, 11.372, 11.40):
    q = A.compute(v, p_post=p, use_paper_dv=False)
    print(p, round(q['dP_min'], 1), round(q['dv_mm_s'], 2), round(q['beta'], 2))

## 4. β가 3이 되는 밀도
밀도를 바꿔 가며 찾아 보자.

In [ ]:
lo, hi = 1000, 4000
for _ in range(40):
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if r['beta_fn'](mid) < 3 else (lo, mid)
print(round(lo), 'kg/m³')